# Hand-Object Interaction Manual Annotation Validation

This notebook aligns manual action segment annotations with the Hand-Object Interaction Feature Layer outputs. It is meant as a diagnostic evaluation: manual segments describe the action step, while many annotation `object` and `hand` fields are `unknown`, and the object detector only has COCO-like object classes. For that reason the notebook separates automatic object-hint checks from broader plausibility summaries.

## What This Checks

- For every manual action segment, summarize left/right active object and temporal proxy objects.
- Compare the top predicted objects against simple action-derived object hints when the action name contains an object that COCO can represent.
- Report score, phase, support ratio, hand activity, and object stability per action segment.
- Highlight segments that look suspicious: low interaction score, no stable object, or mismatch with expected object hints.
- Compare `1s` and `0.5s` temporal windows.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATASET_ROOT = Path("/home/julius/datasets/seminar_object_detection")
EXPORT_ROOT = DATASET_ROOT / "iAssistADL_dataset_export_2026-05-18"
ANNOTATION_ROOT = EXPORT_ROOT / "annotations"
HOI_ROOT = DATASET_ROOT / "upper_feature_outcome" / "hand_object_interaction_features"
PAIR_ROOT = HOI_ROOT / "pair_features"
FRAME_ROOT = HOI_ROOT / "frame_features"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True

ANNOTATION_ROOT

## Configuration

By default this evaluates one mixed-task recording per participant, so the summary is not dominated by one person. Set `MAX_RECORDINGS_PER_PARTICIPANT = None` to evaluate all matching recordings.

In [ ]:
RECORDING_FILTER = "Chaos_several_Tasks_combine"
PARTICIPANTS = None  # Example: ["IA_4_H_03", "IA_4_H_14"]
MAX_RECORDINGS_PER_PARTICIPANT = 1

WINDOWS = {
    "active": {
        "L_object": "L_active_object",
        "R_object": "R_active_object",
        "L_score": "L_interaction_score",
        "R_score": "R_interaction_score",
        "L_phase": "L_interaction_phase",
        "R_phase": "R_interaction_phase",
        "L_support": None,
        "R_support": None,
    },
    "1s": {
        "L_object": "L_temporal_object",
        "R_object": "R_temporal_object",
        "L_score": "L_temporal_score",
        "R_score": "R_temporal_score",
        "L_phase": "L_temporal_phase",
        "R_phase": "R_temporal_phase",
        "L_support": "L_temporal_support_ratio",
        "R_support": "R_temporal_support_ratio",
    },
    "0.5s": {
        "L_object": "L_temporal_0p5_object",
        "R_object": "R_temporal_0p5_object",
        "L_score": "L_temporal_0p5_score",
        "R_score": "R_temporal_0p5_score",
        "L_phase": "L_temporal_0p5_phase",
        "R_phase": "R_temporal_0p5_phase",
        "L_support": "L_temporal_0p5_support_ratio",
        "R_support": "R_temporal_0p5_support_ratio",
    },
}

MIN_GOOD_SCORE = 0.35
MIN_GOOD_SUPPORT = 0.35
MIN_EXPECTED_OBJECT_COVERAGE = 0.20

WINDOWS.keys()

## Object Hints From Action Names

These rules are intentionally conservative. They only score actions whose step names mention objects that the current detector can plausibly output. Actions involving keys, wallets, pens, envelopes, stamps, deodorant, sunglasses, lip balm, wrappers, and similar non-COCO items are still summarized, but not treated as automatic mismatches.

In [ ]:
OBJECT_HINT_RULES = [
    (r"bottle|cap", {"bottle"}),
    (r"glass", {"cup", "wine glass", "bottle"}),
    (r"cup|tea|kettle|water|sugar|drink", {"cup", "bottle"}),
    (r"fork", {"fork"}),
    (r"knife", {"knife"}),
    (r"spoon|ladle|stir|scoop", {"spoon"}),
    (r"cutlery|cut", {"fork", "knife", "spoon"}),
    (r"bowl|pasta|tupper|container", {"bowl"}),
    (r"plate", {"bowl", "dining table"}),
    (r"bar|eat|mouth", {"sandwich", "fork", "spoon", "cup", "bottle"}),
]

def expected_objects_for_step(step):
    text = str(step).lower()
    expected = set()
    for pattern, labels in OBJECT_HINT_RULES:
        if re.search(pattern, text):
            expected |= labels
    return sorted(expected)

for step in ["Take bottle", "Pour pasta into large white bowl", "Pick up pen", "Bring fork to mouth (eat)"]:
    print(step, "->", expected_objects_for_step(step))

## Discover Matching Recordings

In [ ]:
def discover_recordings():
    rows = []
    for segment_path in sorted(ANNOTATION_ROOT.glob("IA_4_H_*/*.segments.csv")):
        full_id = segment_path.name.replace(".segments.csv", "")
        participant_id = segment_path.parent.name
        if RECORDING_FILTER and RECORDING_FILTER not in full_id:
            continue
        if PARTICIPANTS and participant_id not in PARTICIPANTS:
            continue
        frame_path = FRAME_ROOT / participant_id / f"{full_id}_hand_object_frame_features.csv"
        pair_path = PAIR_ROOT / participant_id / f"{full_id}_hand_object_pair_features.csv"
        rows.append({
            "participant_id": participant_id,
            "full_id": full_id,
            "segment_path": segment_path,
            "frame_path": frame_path,
            "pair_path": pair_path,
            "has_frame_features": frame_path.exists(),
            "has_pair_features": pair_path.exists(),
        })
    recordings = pd.DataFrame(rows)
    if recordings.empty:
        raise FileNotFoundError(f"No annotation segments found under {ANNOTATION_ROOT}")
    recordings = recordings.sort_values(["participant_id", "full_id"]).reset_index(drop=True)
    if MAX_RECORDINGS_PER_PARTICIPANT is not None:
        recordings = (
            recordings
            .groupby("participant_id", as_index=False, sort=False)
            .head(MAX_RECORDINGS_PER_PARTICIPANT)
            .reset_index(drop=True)
        )
    return recordings

recordings = discover_recordings()
display(recordings)
missing = recordings[~recordings["has_frame_features"] | ~recordings["has_pair_features"]]
if len(missing):
    warnings.warn(f"{len(missing)} selected recordings are missing HOI feature files and will be skipped.")

## Segment-Level Summary Functions

In [ ]:
def normalize_object(value):
    if pd.isna(value):
        return "none"
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "unknown"}:
        return "none"
    return text

def object_distribution(values):
    series = pd.Series(values).map(normalize_object)
    series = series[series.ne("none")]
    if series.empty:
        return "none"
    counts = series.value_counts(normalize=True).head(4)
    return "; ".join(f"{obj}:{share:.0%}" for obj, share in counts.items())

def phase_distribution(values):
    series = pd.Series(values).dropna().astype(str)
    series = series[~series.str.lower().isin(["none", "nan", "0"])]
    if series.empty:
        return "none"
    counts = series.value_counts(normalize=True).head(4)
    return "; ".join(f"{phase}:{share:.0%}" for phase, share in counts.items())

def expected_coverage(frame_rows, window_spec, expected):
    if not expected or frame_rows.empty:
        return np.nan
    left = frame_rows[window_spec["L_object"]].map(normalize_object)
    right = frame_rows[window_spec["R_object"]].map(normalize_object)
    hit = left.isin(expected) | right.isin(expected)
    return float(hit.mean())

def dominant_object(frame_rows, window_spec):
    objects = pd.concat([
        frame_rows[window_spec["L_object"]].map(normalize_object),
        frame_rows[window_spec["R_object"]].map(normalize_object),
    ], ignore_index=True)
    objects = objects[objects.ne("none")]
    if objects.empty:
        return "none", 0.0
    shares = objects.value_counts(normalize=True)
    return str(shares.index[0]), float(shares.iloc[0])

def summarize_segment(recording, segment, frame_df, window_name, window_spec):
    start_frame = int(segment["start_frame"])
    end_frame = int(segment["end_frame"])
    rows = frame_df[(frame_df["frame_id"] >= start_frame) & (frame_df["frame_id"] <= end_frame)].copy()
    expected = expected_objects_for_step(segment["step"])
    dom_obj, dom_share = dominant_object(rows, window_spec)
    l_score = pd.to_numeric(rows[window_spec["L_score"]], errors="coerce") if len(rows) else pd.Series(dtype=float)
    r_score = pd.to_numeric(rows[window_spec["R_score"]], errors="coerce") if len(rows) else pd.Series(dtype=float)
    best_score = pd.concat([l_score, r_score], ignore_index=True).max() if len(rows) else np.nan
    mean_best_score = pd.concat([l_score, r_score], axis=1).max(axis=1).mean() if len(rows) else np.nan

    support_cols = [window_spec["L_support"], window_spec["R_support"]]
    support_cols = [col for col in support_cols if col]
    mean_support = np.nan
    if support_cols and len(rows):
        support_values = rows[support_cols].apply(pd.to_numeric, errors="coerce")
        mean_support = support_values.max(axis=1).mean()

    coverage = expected_coverage(rows, window_spec, expected)
    phase_cols = [window_spec["L_phase"], window_spec["R_phase"]]
    object_cols = [window_spec["L_object"], window_spec["R_object"]]
    verdict = "diagnostic_only"
    if expected:
        if pd.notna(coverage) and coverage >= MIN_EXPECTED_OBJECT_COVERAGE:
            verdict = "expected_object_seen"
        elif pd.notna(mean_best_score) and mean_best_score >= MIN_GOOD_SCORE:
            verdict = "active_but_unexpected_object"
        else:
            verdict = "weak_or_missing_expected_object"
    elif pd.notna(mean_best_score) and mean_best_score < MIN_GOOD_SCORE:
        verdict = "low_score_diagnostic"

    return {
        "participant_id": recording["participant_id"],
        "full_id": recording["full_id"],
        "window": window_name,
        "step": segment["step"],
        "start_frame": start_frame,
        "end_frame": end_frame,
        "duration_s": float(segment["end_s"]) - float(segment["start_s"]),
        "frames_with_features": int(len(rows)),
        "expected_objects": ", ".join(expected) if expected else "",
        "expected_object_coverage": coverage,
        "dominant_object": dom_obj,
        "dominant_object_share": dom_share,
        "mean_best_score": mean_best_score,
        "max_score": best_score,
        "mean_support_ratio": mean_support,
        "left_objects": object_distribution(rows[object_cols[0]]) if len(rows) else "none",
        "right_objects": object_distribution(rows[object_cols[1]]) if len(rows) else "none",
        "phase_distribution": phase_distribution(pd.concat([rows[phase_cols[0]], rows[phase_cols[1]]], ignore_index=True)) if len(rows) else "none",
        "verdict": verdict,
    }

def summarize_recording(recording):
    segments = pd.read_csv(recording["segment_path"])
    frame_df = pd.read_csv(recording["frame_path"])
    frame_df["frame_id"] = frame_df["frame_id"].astype(int)
    summaries = []
    for _, segment in segments.iterrows():
        for window_name, window_spec in WINDOWS.items():
            summaries.append(summarize_segment(recording, segment, frame_df, window_name, window_spec))
    return pd.DataFrame(summaries)

selected = recordings[recordings["has_frame_features"]].copy()
segment_summary = pd.concat([summarize_recording(row) for _, row in selected.iterrows()], ignore_index=True)
display(segment_summary.head(20))
segment_summary.shape

## Overall Verdict Summary

In [ ]:
verdict_summary = (
    segment_summary
    .groupby(["window", "verdict"], dropna=False)
    .size()
    .rename("segments")
    .reset_index()
)
display(verdict_summary)

score_summary = (
    segment_summary
    .groupby("window")
    .agg(
        segments=("step", "size"),
        mean_best_score=("mean_best_score", "mean"),
        median_best_score=("mean_best_score", "median"),
        mean_support_ratio=("mean_support_ratio", "mean"),
        expected_object_coverage=("expected_object_coverage", "mean"),
    )
    .reset_index()
)
display(score_summary)

In [ ]:
plot_df = segment_summary.copy()
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_df.boxplot(column="mean_best_score", by="window", ax=axes[0])
axes[0].set_title("Mean Best HOI Score by Window")
axes[0].set_xlabel("")
axes[0].set_ylabel("score")

plot_df.boxplot(column="dominant_object_share", by="window", ax=axes[1])
axes[1].set_title("Dominant Object Stability")
axes[1].set_xlabel("")
axes[1].set_ylabel("share")

plot_df.dropna(subset=["expected_object_coverage"]).boxplot(column="expected_object_coverage", by="window", ax=axes[2])
axes[2].set_title("Expected Object Coverage")
axes[2].set_xlabel("")
axes[2].set_ylabel("frame share")
plt.suptitle("")
plt.tight_layout()
plt.show()

## Suspicious Segments To Inspect

In [ ]:
suspicious = segment_summary[
    segment_summary["verdict"].isin([
        "weak_or_missing_expected_object",
        "active_but_unexpected_object",
        "low_score_diagnostic",
    ])
].copy()

suspicious = suspicious.sort_values([
    "verdict",
    "expected_object_coverage",
    "mean_best_score",
], ascending=[True, True, True])

cols = [
    "participant_id", "full_id", "window", "step", "start_frame", "end_frame",
    "expected_objects", "expected_object_coverage", "dominant_object", "dominant_object_share",
    "mean_best_score", "mean_support_ratio", "left_objects", "right_objects", "phase_distribution", "verdict",
]
display(suspicious[cols].head(60))

## Per-Step Behavior Across People

This table helps judge whether similar manual actions lead to similar HOI outputs. Stable repeated actions should generally show similar dominant objects and phases, though non-COCO objects will often be routed to nearby proxy objects.

In [ ]:
per_step = (
    segment_summary
    .groupby(["window", "step"], dropna=False)
    .agg(
        segments=("step", "size"),
        expected_objects=("expected_objects", lambda s: next((x for x in s if x), "")),
        mean_score=("mean_best_score", "mean"),
        mean_support=("mean_support_ratio", "mean"),
        expected_coverage=("expected_object_coverage", "mean"),
        common_dominant_objects=("dominant_object", lambda s: "; ".join(f"{k}:{v}" for k, v in s.value_counts().head(4).items())),
        verdicts=("verdict", lambda s: "; ".join(f"{k}:{v}" for k, v in s.value_counts().items())),
    )
    .reset_index()
    .sort_values(["window", "segments", "step"], ascending=[True, False, True])
)
display(per_step.head(80))

## Single Recording Timeline View

Use this to inspect whether predicted interaction scores rise and objects switch at reasonable manual action boundaries.

In [ ]:
TIMELINE_FULL_ID = selected.iloc[0]["full_id"]
TIMELINE_WINDOW = "1s"  # "active", "1s", or "0.5s"

timeline_recording = selected[selected["full_id"].eq(TIMELINE_FULL_ID)].iloc[0]
timeline_segments = pd.read_csv(timeline_recording["segment_path"])
timeline_frames = pd.read_csv(timeline_recording["frame_path"])
spec = WINDOWS[TIMELINE_WINDOW]

timeline_frames["best_score"] = timeline_frames[[spec["L_score"], spec["R_score"]]].apply(pd.to_numeric, errors="coerce").max(axis=1)
timeline_frames["left_object"] = timeline_frames[spec["L_object"]].map(normalize_object)
timeline_frames["right_object"] = timeline_frames[spec["R_object"]].map(normalize_object)

fig, ax = plt.subplots(figsize=(18, 5))
ax.plot(timeline_frames["frame_id"], timeline_frames["best_score"], label="best left/right HOI score", linewidth=1.5)
for _, seg in timeline_segments.iterrows():
    ax.axvspan(seg["start_frame"], seg["end_frame"], alpha=0.08)
    mid = (seg["start_frame"] + seg["end_frame"]) / 2
    ax.text(mid, ax.get_ylim()[1] * 0.95, str(seg["step"])[:28], rotation=90, va="top", ha="center", fontsize=8)
ax.set_title(f"{TIMELINE_FULL_ID} - {TIMELINE_WINDOW} HOI score over manual steps")
ax.set_xlabel("frame")
ax.set_ylabel("score")
ax.legend()
plt.tight_layout()
plt.show()

## Segment Detail Drill-Down

In [ ]:
DETAIL_FULL_ID = TIMELINE_FULL_ID
DETAIL_STEP_CONTAINS = "bottle"
DETAIL_WINDOW = "1s"

detail_rows = segment_summary[
    segment_summary["full_id"].eq(DETAIL_FULL_ID)
    & segment_summary["window"].eq(DETAIL_WINDOW)
    & segment_summary["step"].str.contains(DETAIL_STEP_CONTAINS, case=False, na=False)
].copy()
display(detail_rows[cols])

if len(detail_rows):
    detail = detail_rows.iloc[0]
    rec = selected[selected["full_id"].eq(DETAIL_FULL_ID)].iloc[0]
    frame_df = pd.read_csv(rec["frame_path"])
    spec = WINDOWS[DETAIL_WINDOW]
    segment_frames = frame_df[(frame_df["frame_id"] >= detail["start_frame"]) & (frame_df["frame_id"] <= detail["end_frame"])].copy()
    display(segment_frames[[
        "frame_id", spec["L_object"], spec["L_score"], spec["L_phase"],
        spec["R_object"], spec["R_score"], spec["R_phase"],
        "two_hand_temporal_object", "two_hand_temporal_score",
    ]].head(50))

## Optional Pair-Level Detail

Pair features can explain why a segment received a particular score: distance, gap, contact duration, motion scores, and phase. This cell shows the top scoring hand-object pairs inside one manual segment.

In [ ]:
if len(detail_rows):
    rec = selected[selected["full_id"].eq(DETAIL_FULL_ID)].iloc[0]
    pair_df = pd.read_csv(rec["pair_path"])
    pair_detail = pair_df[(pair_df["frame_id"] >= detail["start_frame"]) & (pair_df["frame_id"] <= detail["end_frame"])].copy()
    pair_cols = [
        "frame_id", "hand_side", "object_class", "object_id", "interaction_score", "interaction_phase_name",
        "visual_score", "temporal_score", "skeleton_motion_score", "imu_motion_score",
        "center_distance_norm", "bbox_gap_norm", "expanded_iou", "contact_duration_s",
    ]
    available_pair_cols = [col for col in pair_cols if col in pair_detail.columns]
    display(pair_detail.sort_values("interaction_score", ascending=False)[available_pair_cols].head(80))

## Export Tables

In [ ]:
VALIDATION_ROOT = DATASET_ROOT / "upper_feature_outcome" / "hand_object_interaction_validation"
VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

summary_path = VALIDATION_ROOT / f"manual_annotation_segment_summary_{RECORDING_FILTER}.csv"
step_path = VALIDATION_ROOT / f"manual_annotation_step_summary_{RECORDING_FILTER}.csv"
segment_summary.to_csv(summary_path, index=False)
per_step.to_csv(step_path, index=False)

{
    "segment_summary": str(summary_path),
    "step_summary": str(step_path),
}